# Preprocessing design — zaakceptowana supervised sample

**Stała granica próby.** Notebook przyjmuje bez ponownej selekcji:

```text
development 2011–2022
membership_status == eligible
target_status == available
x_t_status in {available_core, partially_available}
N = 23 218; train = 19 671; validation = 3 547
```

Frozen target, universe i raw `X_t` pozostają wejściami tylko do odczytu.
Missingness, outliery i validation diagnostics nie mogą zmienić supervised
sample ani frozen feature blocks `L`, `L+D`, `L+D+R`.

**Zakres.** Projekt obejmuje train-median imputation, missing indicators,
winsoryzację i scaling. Nie projektuje temporal CV, nie trenuje modeli i
nie używa lat 2023–2024. Parametry pokazywane dla validation są zawsze
wcześniej dopasowane wyłącznie na train. W przyszłym CV dokładnie ta sama
operacja `fit` musi odbywać się od nowa wewnątrz każdego train fold.

**Reprodukowalność.** Jedna jawna komórka buduje fixed sample. Kolejne
komórki tworzą nowe tabele bez in-place modyfikacji wejść. Implementacja
używana przez notebook znajduje się w `src/modeling/preprocessing.py`.


In [1]:
from pathlib import Path
import csv
import hashlib
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 72)

DEV_YEAR_MIN, DEV_YEAR_MAX = 2011, 2022
ACCEPTED_X_STATUSES = {"available_core", "partially_available"}

def project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs/x_t_pit_v1_freeze_manifest.yaml").is_file():
            return candidate
    raise FileNotFoundError("Nie znaleziono katalogu głównego projektu.")

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.modeling.preprocessing import (
    FEATURE_BLOCKS,
    FROZEN_BLOCK_COMPARISONS,
    FinancialPreprocessor,
    PreprocessingPolicy,
    features_for_blocks,
)

FEATURES = list(features_for_blocks(("L", "D", "R")))
BLOCK_BY_FEATURE = {
    feature: block for block, features in FEATURE_BLOCKS.items() for feature in features
}
SHORT = {
    "log_assets_t": "log_assets",
    "roa_t": "roa",
    "ocf_to_assets_t": "ocf_assets",
    "current_ratio_t": "current_ratio",
    "liabilities_to_assets_t": "liab_assets",
    "working_capital_to_assets_t": "wc_assets",
    "accruals_to_assets_t": "accruals_assets",
    "asset_growth_1y": "asset_growth",
    "delta_roa_1y": "delta_roa",
    "delta_ocf_to_assets_1y": "delta_ocf_assets",
    "current_ratio_change_1y": "current_ratio_chg",
    "delta_liabilities_to_assets_1y": "delta_liab_assets",
    "log1p_revenues_t": "log_revenues",
    "profit_margin_t": "profit_margin",
    "ocf_margin_t": "ocf_margin",
    "asset_turnover_t": "asset_turnover",
    "revenue_growth_1y": "revenue_growth",
}

PATHS = {
    "universe": ROOT / "data/processed/research_universe_pit.csv",
    "target": ROOT / "data/interim/target_candidate_v2_pit_b.csv",
    "x_t": ROOT / "data/processed/x_t_pit_v1_raw.csv",
    "target_application": ROOT / "data/processed/research_universe_pit_v1_1_0_target_pit_b_v1_0_0.csv",
}
EXPECTED = {
    "universe": ("a449c8145d1f46f954f12b1dfc079bb0b367c4f7f5edf3332a983ad7c1fb8182", 103099, 81),
    "target": ("473aa403dfd15822a15ce985f7698efe4a4e3a66bcf30b7634f0ca646805e0ff", 26917, 802),
    "x_t": ("0f1b35b9ffbb1fb1c1cdfb7dff12e3efd8fb38f60b33407ff2b2a8fb6b88397f", 64901, 1072),
    "target_application": ("ea42eb43018b2c8e238e2c4757260bb692e27edd5429628e28892f360f0f7f7d", 64901, 832),
}

def fingerprint_csv(path: Path) -> dict:
    digest = hashlib.sha256()
    byte_count = 0
    newline_count = 0
    with path.open("rb") as handle:
        while chunk := handle.read(8 * 1024 * 1024):
            digest.update(chunk)
            byte_count += len(chunk)
            newline_count += chunk.count(b"\n")
    with path.open(newline="", encoding="utf-8") as handle:
        columns = len(next(csv.reader(handle)))
    return {
        "sha256": digest.hexdigest(),
        "rows": max(newline_count - 1, 0),
        "columns": columns,
        "MiB": round(byte_count / 1024**2, 1),
    }

def read_development(path: Path, usecols: list[str]) -> pd.DataFrame:
    parts = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=10_000, low_memory=False):
        years = pd.to_numeric(chunk["feature_year"], errors="coerce")
        parts.append(chunk.loc[years.between(DEV_YEAR_MIN, DEV_YEAR_MAX)].copy())
    return pd.concat(parts, ignore_index=True)

def build_fixed_sample() -> tuple[pd.DataFrame, pd.DataFrame]:
    x_columns = [
        "research_universe_company_year_id", "feature_year", "split",
        "membership_status", "x_t_status", "L_available_count",
        "D_available_count", "R_available_count", "feature_available_count",
    ]
    for feature in FEATURES:
        x_columns.extend([f"{feature}_value", f"{feature}_status"])
    target_columns = [
        "research_universe_company_year_id", "feature_year", "target_status"
    ]
    x_frame = read_development(PATHS["x_t"], x_columns)
    target_frame = read_development(PATHS["target_application"], target_columns).rename(
        columns={"feature_year": "target_feature_year"}
    )
    joined = x_frame.merge(
        target_frame,
        on="research_universe_company_year_id",
        how="left",
        validate="one_to_one",
    )
    fixed = joined.loc[
        joined["membership_status"].eq("eligible")
        & joined["target_status"].eq("available")
        & joined["x_t_status"].isin(ACCEPTED_X_STATUSES)
    ].copy()
    assert joined["feature_year"].eq(joined["target_feature_year"]).all()
    assert fixed["feature_year"].between(2011, 2022).all()
    assert len(fixed) == 23_218
    assert fixed["split"].value_counts().to_dict() == {
        "train": 19_671, "validation": 3_547
    }
    assert fixed["research_universe_company_year_id"].is_unique
    assert fixed["membership_status"].eq("eligible").all()
    assert fixed["target_status"].eq("available").all()
    assert set(fixed["x_t_status"].unique()) == ACCEPTED_X_STATUSES

    values = fixed[[f"{feature}_value" for feature in FEATURES]].rename(
        columns={f"{feature}_value": feature for feature in FEATURES}
    )
    values = values.apply(pd.to_numeric, errors="raise").astype(float)
    for feature in FEATURES:
        status_available = fixed[f"{feature}_status"].eq("available")
        assert status_available.eq(values[feature].notna()).all(), feature
    assert not np.isinf(values.to_numpy()).any()
    assert fixed["feature_available_count"].eq(values.notna().sum(axis=1)).all()
    values.index = fixed.index
    return fixed, values

def show(title: str, frame: pd.DataFrame | pd.Series) -> None:
    print(f"\n{title}")
    print("=" * len(title))
    print(frame.to_string())

fingerprints = {name: fingerprint_csv(path) for name, path in PATHS.items()}
for name, actual in fingerprints.items():
    expected_hash, expected_rows, expected_columns = EXPECTED[name]
    assert actual["sha256"] == expected_hash, f"Hash mismatch: {name}"
    assert actual["rows"] == expected_rows, f"Row mismatch: {name}"
    assert actual["columns"] == expected_columns, f"Column mismatch: {name}"

sample_frame, feature_matrix = build_fixed_sample()
train_mask = sample_frame["split"].eq("train")
validation_mask = sample_frame["split"].eq("validation")
train_X = feature_matrix.loc[train_mask].copy()
validation_X = feature_matrix.loc[validation_mask].copy()

integrity = pd.DataFrame.from_dict(fingerprints, orient="index")
integrity["sha256_ok"] = True
integrity["shape_ok"] = True
show("Frozen input integrity", integrity[["rows", "columns", "MiB", "sha256_ok", "shape_ok"]])
print(
    f"\nFixed sample verified: N={len(sample_frame):,}; "
    f"train={len(train_X):,}; validation={len(validation_X):,}; "
    f"years={sample_frame.feature_year.min()}–{sample_frame.feature_year.max()}"
)
print(f"Preprocessing module: {ROOT / 'src/modeling/preprocessing.py'}")



Frozen input integrity
                      rows  columns    MiB  sha256_ok  shape_ok
universe            103099       81  106.4       True      True
target               26917      802  169.3       True      True
x_t                  64901     1072  776.7       True      True
target_application   64901      832  383.7       True      True

Fixed sample verified: N=23,218; train=19,671; validation=3,547; years=2011–2022
Preprocessing module: /Users/oskarstachowski/qnn-financial-statement-analysis/src/modeling/preprocessing.py


## 1. Struktura missingness w zaakceptowanej próbie

Liczby dostępnych cech są raportowane osobno dla L (0–7), D (0–5) i R
(0–5), a następnie łącznie (1–17). Zera w tabelach są zachowane jawnie.
`partially_available` nie jest filtrowane według minimalnej liczby cech.


In [2]:
def count_distribution(
    frame: pd.DataFrame, column: str, possible_values: range
) -> pd.DataFrame:
    counts = pd.crosstab(frame[column], frame["split"]).reindex(
        index=list(possible_values), columns=["train", "validation"], fill_value=0
    )
    counts["all"] = counts.sum(axis=1)
    counts["train_pct"] = (100 * counts["train"] / counts["train"].sum()).round(2)
    counts["validation_pct"] = (
        100 * counts["validation"] / counts["validation"].sum()
    ).round(2)
    counts["all_pct"] = (100 * counts["all"] / counts["all"].sum()).round(2)
    return counts

block_distributions = {
    "L": count_distribution(sample_frame, "L_available_count", range(0, 8)),
    "D": count_distribution(sample_frame, "D_available_count", range(0, 6)),
    "R": count_distribution(sample_frame, "R_available_count", range(0, 6)),
}
partial_frame = sample_frame.loc[
    sample_frame["x_t_status"].eq("partially_available")
].copy()
partial_block_distributions = {
    "L": count_distribution(partial_frame, "L_available_count", range(0, 8)),
    "D": count_distribution(partial_frame, "D_available_count", range(0, 6)),
    "R": count_distribution(partial_frame, "R_available_count", range(0, 6)),
}

total_available = feature_matrix.notna().sum(axis=1).rename("available_features")
total_frame = sample_frame[["split", "x_t_status"]].assign(
    available_features=total_available
)
total_distribution = count_distribution(total_frame, "available_features", range(1, 18))
partial_total_distribution = count_distribution(
    total_frame.loc[total_frame["x_t_status"].eq("partially_available")],
    "available_features",
    range(1, 18),
)

for block in ("L", "D", "R"):
    show(f"{block}: liczba dostępnych cech — cała fixed sample", block_distributions[block])
for block in ("L", "D", "R"):
    show(
        f"{block}: liczba dostępnych cech — partially_available only",
        partial_block_distributions[block],
    )
show("Łączna liczba dostępnych cech 1–17 — cała fixed sample", total_distribution)
show("Łączna liczba dostępnych cech 1–17 — partially_available", partial_total_distribution)



L: liczba dostępnych cech — cała fixed sample
split              train  validation    all  train_pct  validation_pct  all_pct
L_available_count                                                              
0                      3           1      4       0.02            0.03     0.02
1                      3           0      3       0.02            0.00     0.01
2                      2           0      2       0.01            0.00     0.01
3                      0           0      0       0.00            0.00     0.00
4                    257           3    260       1.31            0.08     1.12
5                    221          19    240       1.12            0.54     1.03
6                     26           4     30       0.13            0.11     0.13
7                  19159        3520  22679      97.40           99.24    97.68

D: liczba dostępnych cech — cała fixed sample
split              train  validation    all  train_pct  validation_pct  all_pct
D_available_count         

### Missingness per feature i różnice train–validation

Brak oznacza `feature_status != available`, co w frozen raw `X_t` jest
równoważne `value = NA`. Severity jest opisowe, nie jest regułą usuwania:
`<5% low`, `5–15% moderate`, `15–30% high`, `>=30% very high`.


In [3]:
missing_rows = []
for feature in FEATURES:
    row = {"block": BLOCK_BY_FEATURE[feature], "feature": feature}
    for split, matrix in (("train", train_X), ("validation", validation_X)):
        missing = matrix[feature].isna()
        row[f"{split}_missing_n"] = int(missing.sum())
        row[f"{split}_missing_pct"] = round(100 * missing.mean(), 2)
    row["validation_minus_train_pp"] = round(
        row["validation_missing_pct"] - row["train_missing_pct"], 2
    )
    train_rate = row["train_missing_pct"]
    row["severity_train"] = (
        "very_high" if train_rate >= 30
        else "high" if train_rate >= 15
        else "moderate" if train_rate >= 5
        else "low"
    )
    missing_rows.append(row)
missingness = pd.DataFrame(missing_rows).set_index(["block", "feature"])
severity_counts = (
    missingness.reset_index().groupby(["severity_train", "block"]).size()
    .unstack(fill_value=0)
    .reindex(["low", "moderate", "high", "very_high"], fill_value=0)
)
show("Missingness per feature", missingness)
show("Liczba cech według severity na train", severity_counts)
print(
    "\nFeatures with >=30% train missingness: ",
    missingness.loc[missingness["train_missing_pct"].ge(30)].index.tolist(),
)



Missingness per feature
                                      train_missing_n  train_missing_pct  validation_missing_n  validation_missing_pct  validation_minus_train_pp severity_train
block feature                                                                                                                                                   
L     log_assets_t                                  5               0.03                     1                    0.03                       0.00            low
      roa_t                                       385               1.96                    10                    0.28                      -1.68            low
      ocf_to_assets_t                             295               1.50                     6                    0.17                      -1.33            low
      current_ratio_t                              75               0.38                    12                    0.34                      -0.04            low
      lia

### Współwystępowanie braków

Phi jest korelacją dwóch binarnych masek missingness; Jaccard to udział
wspólnych braków w unii braków. Obie miary są liczone osobno dla train i
validation. Polityka preprocessingu jest ustalona niezależnie od tych
validation diagnostics.


In [4]:
def missing_pair_diagnostics(matrix: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    missing = matrix.isna().rename(columns=SHORT)
    joint_counts = missing.astype(int).T.dot(missing.astype(int))
    phi = missing.astype(float).corr()
    pairs = []
    columns = list(missing.columns)
    for left_index, left in enumerate(columns):
        for right in columns[left_index + 1:]:
            left_mask = missing[left]
            right_mask = missing[right]
            joint = int((left_mask & right_mask).sum())
            union = int((left_mask | right_mask).sum())
            pairs.append(
                {
                    "feature_1": left,
                    "feature_2": right,
                    "joint_missing_n": joint,
                    "jaccard": joint / union if union else np.nan,
                    "phi": phi.loc[left, right],
                }
            )
    pair_table = pd.DataFrame(pairs).sort_values(
        ["phi", "joint_missing_n"], ascending=False
    )
    return joint_counts, phi, pair_table

train_joint, train_phi, train_pairs = missing_pair_diagnostics(train_X)
validation_joint, validation_phi, validation_pairs = missing_pair_diagnostics(validation_X)
show("Train: joint missing counts matrix", train_joint)
show("Train: phi missingness matrix", train_phi.round(2))
show("Train: top 15 par współwystępujących braków", train_pairs.head(15).round(4).set_index(["feature_1", "feature_2"]))
show("Validation: top 15 par współwystępujących braków", validation_pairs.head(15).round(4).set_index(["feature_1", "feature_2"]))



Train: joint missing counts matrix
                   log_assets  roa  ocf_assets  current_ratio  liab_assets  wc_assets  accruals_assets  asset_growth  delta_roa  delta_ocf_assets  current_ratio_chg  delta_liab_assets  log_revenues  profit_margin  ocf_margin  asset_turnover  revenue_growth
log_assets                  5    5           5              3            5          5                5             5          5                 5                  3                  5             0              0           0               5               1
roa                         5  385         263              5            6          7              385           260        385               281                259                261           265            380         265             270             283
ocf_assets                  5  263         295              6            7          8              295           247        269               295                246                248          

**Wniosek o missingness.** Braki mają wyraźną strukturę blokową:
current revenue features znikają niemal razem, podobnie dynamiczne
pary oparte na comparative `t-1`. Nie są więc wiarygodnie MCAR.
Complete-case filter usuwałby całe profile spółek i filingów, zamiast
niezależnych losowych komórek.

## 2. Rozkłady, outliery i proponowane granice winsoryzacji

Polityka jest ustalona **przed** oglądaniem wyników modeli:

- granica dolna = train 1st percentile per feature;
- granica górna = train 99th percentile per feature;
- kwantyle liczone tylko z obserwowanych, skończonych wartości train;
- validation jest jedynie clippingowane gotowymi granicami;
- brak usuwania wierszy i brak winsoryzacji missing indicators.

Tabela kwantyli jest odpowiednikiem diagnostyki rozkładu w formie
reprodukowalnej i dokładniejszej od osi wykresu przy ekstremalnych
ogonach ratio.


In [5]:
POLICY = PreprocessingPolicy(
    lower_quantile=0.01,
    upper_quantile=0.99,
    add_missing_indicators=True,
    near_constant_dominant_share=0.995,
)
full_preprocessor = FinancialPreprocessor.for_blocks(
    ("L", "D", "R"), policy=POLICY
).fit(train_X)

distribution_rows = []
for feature in FEATURES:
    train_values = train_X[feature].dropna()
    quantiles = train_values.quantile(
        [0.0, 0.001, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999, 1.0]
    )
    iqr = quantiles.loc[0.75] - quantiles.loc[0.25]
    iqr_lower = quantiles.loc[0.25] - 1.5 * iqr
    iqr_upper = quantiles.loc[0.75] + 1.5 * iqr
    train_clipped = (
        train_values.lt(full_preprocessor.lower_bounds_[feature])
        | train_values.gt(full_preprocessor.upper_bounds_[feature])
    )
    validation_values = validation_X[feature].dropna()
    validation_clipped = (
        validation_values.lt(full_preprocessor.lower_bounds_[feature])
        | validation_values.gt(full_preprocessor.upper_bounds_[feature])
    )
    distribution_rows.append(
        {
            "block": BLOCK_BY_FEATURE[feature],
            "feature": feature,
            "observed_train_n": len(train_values),
            "min": quantiles.loc[0.0],
            "p0.1": quantiles.loc[0.001],
            "p1_winsor_lower": quantiles.loc[0.01],
            "p5": quantiles.loc[0.05],
            "p25": quantiles.loc[0.25],
            "median": quantiles.loc[0.50],
            "p75": quantiles.loc[0.75],
            "p95": quantiles.loc[0.95],
            "p99_winsor_upper": quantiles.loc[0.99],
            "p99.9": quantiles.loc[0.999],
            "max": quantiles.loc[1.0],
            "IQR_outlier_pct": 100 * (
                train_values.lt(iqr_lower) | train_values.gt(iqr_upper)
            ).mean(),
            "train_clipped_n": int(train_clipped.sum()),
            "train_clipped_pct": 100 * train_clipped.mean(),
            "validation_clipped_n": int(validation_clipped.sum()),
            "validation_clipped_pct": 100 * validation_clipped.mean(),
        }
    )
distributions = pd.DataFrame(distribution_rows).set_index(["block", "feature"])
show("Train quantile distributions, outliers and fixed winsor bounds", distributions.round(4))



Train quantile distributions, outliers and fixed winsor bounds
                                      observed_train_n         min       p0.1  p1_winsor_lower       p5      p25   median      p75      p95  p99_winsor_upper     p99.9         max  IQR_outlier_pct  train_clipped_n  train_clipped_pct  validation_clipped_n  validation_clipped_pct
block feature                                                                                                                                                                                                                                                                         
L     log_assets_t                               19666      7.1601     8.8992          11.2391  13.8079  17.5629  19.7535  21.4600  23.5392           24.9492   26.1481     26.8193           1.3984              394             2.0035                    69                  1.9459
      roa_t                                      19286 -11072.3464  -147.9815         -15.8314  -2.

## 3. Train-only imputation, indicators i scaling — diagnostyka

Transformacja ma kolejność:

```text
raw NA mask
  → clip observed values do train p1/p99
  → fill NA train median per feature
  → StandardScaler(mean, population std) fit na przetworzonym train
  → append binary feature__missing indicators bez skalowania
```

Mediana zawsze leży wewnątrz granic p1/p99, dlatego median imputation
nie jest zniekształcana przez clipping. Brak cechy całkowicie missing w
train jest błędem blokującym, a nie sygnałem do wstawienia zera.


In [6]:
train_stages = full_preprocessor.transform_stages(train_X)
validation_stages = full_preprocessor.transform_stages(validation_X)

impact_rows = []
indicator_rows = []
stability_rows = []
scaling_rows = []
for feature in FEATURES:
    raw_observed = train_stages["raw"][feature].dropna()
    winsor_observed = train_stages["winsorized"][feature].dropna()
    imputed = train_stages["imputed"][feature]
    scaled_train = train_stages["scaled"][feature]
    scaled_validation = validation_stages["scaled"][feature]
    indicator_name = f"{feature}__missing"
    train_indicator = train_stages["indicators"][indicator_name]
    validation_indicator = validation_stages["indicators"][indicator_name]

    impact_rows.append(
        {
            "block": BLOCK_BY_FEATURE[feature],
            "feature": feature,
            "raw_observed_mean": raw_observed.mean(),
            "raw_observed_sd": raw_observed.std(ddof=0),
            "winsor_observed_mean": winsor_observed.mean(),
            "winsor_observed_sd": winsor_observed.std(ddof=0),
            "train_median_imputed": full_preprocessor.medians_[feature],
            "post_imputation_mean": imputed.mean(),
            "post_imputation_sd": imputed.std(ddof=0),
            "imputed_from_missing_pct": 100 * train_indicator.mean(),
        }
    )
    dominant_indicator_share = train_indicator.value_counts(normalize=True).max()
    indicator_rows.append(
        {
            "block": BLOCK_BY_FEATURE[feature],
            "indicator": indicator_name,
            "train_missing_n": int(train_indicator.sum()),
            "train_missing_pct": 100 * train_indicator.mean(),
            "validation_missing_n": int(validation_indicator.sum()),
            "validation_missing_pct": 100 * validation_indicator.mean(),
            "dominant_value_share_pct": 100 * dominant_indicator_share,
            "constant_train": train_indicator.nunique(dropna=False) == 1,
            "near_constant_train": dominant_indicator_share >= 0.995,
        }
    )
    dominant_financial_share = imputed.value_counts(normalize=True).max()
    stability_rows.append(
        {
            "block": BLOCK_BY_FEATURE[feature],
            "feature": feature,
            "unique_after_imputation": imputed.nunique(dropna=False),
            "dominant_value_share_pct": 100 * dominant_financial_share,
            "variance_after_imputation": imputed.var(ddof=0),
            "constant": feature in full_preprocessor.constant_features_,
            "near_constant": feature in full_preprocessor.near_constant_features_,
        }
    )
    scaling_rows.append(
        {
            "block": BLOCK_BY_FEATURE[feature],
            "feature": feature,
            "train_scaled_mean": scaled_train.mean(),
            "train_scaled_sd": scaled_train.std(ddof=0),
            "train_scaled_min": scaled_train.min(),
            "train_scaled_max": scaled_train.max(),
            "validation_scaled_mean": scaled_validation.mean(),
            "validation_scaled_sd": scaled_validation.std(ddof=0),
        }
    )

imputation_impact = pd.DataFrame(impact_rows).set_index(["block", "feature"])
indicators = pd.DataFrame(indicator_rows).set_index(["block", "indicator"])
financial_stability = pd.DataFrame(stability_rows).set_index(["block", "feature"])
scaling_diagnostics = pd.DataFrame(scaling_rows).set_index(["block", "feature"])

assert len(train_stages["transformed"]) == 19_671
assert len(validation_stages["transformed"]) == 3_547
assert train_stages["transformed"].shape[1] == 34
assert validation_stages["transformed"].shape[1] == 34
assert np.isfinite(train_stages["transformed"].to_numpy()).all()
assert np.isfinite(validation_stages["transformed"].to_numpy()).all()

show("Wpływ winsoryzacji i median imputation na train distributions", imputation_impact.round(5))
show("Missing indicators — liczba i częstość", indicators.round(4))
show("Stałość frozen financial features po imputacji", financial_stability.round(6))
show("Scaling diagnostics; validation transform bez refit", scaling_diagnostics.round(4))



Wpływ winsoryzacji i median imputation na train distributions
                                      raw_observed_mean  raw_observed_sd  winsor_observed_mean  winsor_observed_sd  train_median_imputed  post_imputation_mean  post_imputation_sd  imputed_from_missing_pct
block feature                                                                                                                                                                                               
L     log_assets_t                             19.35078          2.95314              19.35805             2.90564              19.75354              19.35815             2.90527                   0.02542
      roa_t                                    -1.68933         83.18699              -0.45780             2.00576               0.01838              -0.44848             1.98713                   1.95720
      ocf_to_assets_t                          -0.25505          6.88305              -0.11360             0.67418   

**Wynik diagnostyki stałości.** Żadna z 17 frozen financial features nie
staje się stała ani prawie stała po winsoryzacji i imputacji. Żaden
missing indicator nie jest stały na pełnym train. Cztery wskaźniki są
rzadkie według jawnego progu dominant share ≥99,5%:
`log_assets_t__missing`, `current_ratio_t__missing`,
`liabilities_to_assets_t__missing` i
`working_capital_to_assets_t__missing`. Pozostają w głównym schemacie,
ponieważ ich usunięcie byłoby data-driven zmianą reprezentacji, a stały
zestaw indicators pozwala również oznaczyć brak pojawiający się dopiero
w validation lub przyszłym foldzie.

## 4. Warianty A/B/C i zachowanie frozen blocks

Warianty są porównane konstrukcyjnie, bez wyników validation i bez
modeli. Wariant A zmienia populację między blokami; B i C zachowują
dokładnie tę samą zaakceptowaną sample.


In [7]:
block_rows = []
fitted_by_block = {}
for blocks in FROZEN_BLOCK_COMPARISONS:
    feature_names = features_for_blocks(blocks)
    label = "+".join(blocks)
    complete_mask = feature_matrix.loc[:, feature_names].notna().all(axis=1)
    transformer_c = FinancialPreprocessor.for_blocks(blocks, policy=POLICY).fit(
        train_X.loc[:, feature_names]
    )
    transformer_b = FinancialPreprocessor.for_blocks(
        blocks,
        policy=PreprocessingPolicy(
            lower_quantile=0.01,
            upper_quantile=0.99,
            add_missing_indicators=False,
        ),
    ).fit(train_X.loc[:, feature_names])
    transformed_train_c = transformer_c.transform(train_X.loc[:, feature_names])
    transformed_validation_c = transformer_c.transform(validation_X.loc[:, feature_names])
    transformed_train_b = transformer_b.transform(train_X.loc[:, feature_names])
    fitted_by_block[label] = transformer_c

    assert len(transformed_train_c) == 19_671
    assert len(transformed_validation_c) == 3_547
    assert len(transformed_train_b) == 19_671
    block_rows.append(
        {
            "frozen_block": label,
            "raw_features": len(feature_names),
            "A_complete_case_all_n": int(complete_mask.sum()),
            "A_complete_case_train_n": int((complete_mask & train_mask).sum()),
            "A_complete_case_validation_n": int((complete_mask & validation_mask).sum()),
            "A_retention_pct": 100 * complete_mask.mean(),
            "B_sample_n": len(sample_frame),
            "B_output_columns": transformed_train_b.shape[1],
            "C_sample_n": len(sample_frame),
            "C_output_columns": transformed_train_c.shape[1],
        }
    )
block_comparison = pd.DataFrame(block_rows).set_index("frozen_block")

variants = pd.DataFrame(
    [
        {
            "variant": "A complete cases",
            "rows": "zależne od bloku",
            "missing_signal": "utracony przez row deletion",
            "population_comparability": "nie",
            "rola": "wyłącznie sensitivity / opis bias",
        },
        {
            "variant": "B median, no indicators",
            "rows": "stałe N=23218",
            "missing_signal": "utracony; NA staje się medianą",
            "population_comparability": "tak",
            "rola": "obowiązkowa ablation",
        },
        {
            "variant": "C median + indicators",
            "rows": "stałe N=23218",
            "missing_signal": "zachowany jawnie",
            "population_comparability": "tak",
            "rola": "rekomendowany wariant główny",
        },
    ]
).set_index("variant")

show("A/B/C: row counts i liczba kolumn według frozen block", block_comparison.round(2))
show("Porównanie koncepcyjne A/B/C", variants)



A/B/C: row counts i liczba kolumn według frozen block
              raw_features  A_complete_case_all_n  A_complete_case_train_n  A_complete_case_validation_n  A_retention_pct  B_sample_n  B_output_columns  C_sample_n  C_output_columns
frozen_block                                                                                                                                                                         
L                        7                  22679                    19159                          3520            97.68       23218                 7       23218                14
L+D                     12                  20039                    16962                          3077            86.31       23218                12       23218                24
L+D+R                   17                  18523                    15654                          2869            79.78       23218                17       23218                34

Porównanie koncepcyjne A/B/C
     

## 5. Rekomendowana preprocessing policy

### Imputation

- Median per feature, fit wyłącznie na supervised train.
- W przyszłym CV mediana jest fit od nowa wyłącznie wewnątrz train fold.
- Jedna mediana globalna dla training partition; bez median per rok,
  sektor albo z użyciem validation.
- Feature całkowicie missing w train/fold powoduje jawny błąd i decyzję
  metodologiczną; nie jest automatycznie wypełniana zerem.
- Imputacja nie usuwa wierszy i nie zmienia sample N=23 218.

### Missing indicators

- Rekomendowany wariant główny: **C, median + indicators**.
- Jeden binarny `feature__missing` dla każdej frozen feature obecnej w
  analizowanym bloku, wyliczony przed imputacją.
- Indicators są zawsze obecne w schemacie, także jeśli w konkretnym
  train fold okażą się stałe; są raportowane, nie skalowane i nie
  winsoryzowane.
- Wariant B bez indicators jest obowiązkową ablation ustaloną przed
  modelami. Wariant A complete-case nie jest próbą główną.

### Winsorization

- Per-feature two-sided clipping do **1st/99th percentile train**.
- Kwantyle są fit na observed finite train values przed imputacją.
- Validation/future fold jest wyłącznie transformowany gotowymi caps.
- Brak usuwania outlier rows, brak ręcznych granic dobranych do targetu,
  brak osobnych granic per rok/sektor.
- Raw frozen `X_t` pozostaje bez zmian; caps i liczba clipped values są
  częścią przyszłego fitted preprocessing artifact.

### Scaling

- Standardization `(x - train_mean) / train_population_std` po
  winsoryzacji i imputacji.
- Center i scale są fit wyłącznie na przetworzonym train/fold.
- Skalowane są tylko finansowe features; missing indicators pozostają
  0/1.
- Zerowa wariancja daje audit flag i bezpieczny scale=1, nigdy dzielenie
  przez zero ani automatyczne usunięcie frozen feature.
- Ewentualne późniejsze QNN angle/range encoding jest elementem
  architektury modelu, nie tej ogólnej policy.

### Frozen blocks i wspólna próba

Główne porównania zachowują dokładnie te same 23 218 wierszy:

| Blok | Raw financial features | Wariant C output |
|---|---:|---:|
| L | 7 | 7 scaled + 7 indicators = 14 |
| L+D | 12 | 12 scaled + 12 indicators = 24 |
| L+D+R | 17 | 17 scaled + 17 indicators = 34 |

Brak R nigdy nie usuwa obserwacji z modelu L+D+R.

### Decyzje wymagające zatwierdzenia

1. Zatwierdzić wariant C jako preprocessing główny i B jako obowiązkową
   ablation; A pozostawić wyłącznie jako sensitivity check dotyczący
   complete-case selection bias.
2. Zatwierdzić train 1%/99% jako główne caps. Opcjonalna analiza
   odporności `no winsorization` albo `0.5%/99.5%` musi być ustalona
   przed modelami i nie może być wybierana według validation/test.
3. Zatwierdzić pozostawienie wszystkich indicators, w tym czterech
   rzadkich near-constant indicators, dla stabilnego schematu między
   foldami. Ich ewentualne usunięcie byłoby osobną, train-fold-only
   regułą feature filtering wymagającą prerejestracji.
4. Zatwierdzić StandardScaler po winsoryzacji. RobustScaler może być
   wyłącznie z góry ustalonym sensitivity wariantem, nie wyborem po
   validation performance.

**Nie wykonano CV, treningu, feature selection ani oceny predictive
performance. Test 2023–2024 nie został wczytany do ramki analitycznej.**
